# Képgeneráló alkalmazások építése (Azure OpenAI)

Az LLM-ek többet tudnak, mint csupán szöveg generálása. Szöveges leírásokból képeket is generálhatunk. A képek mint modalitás hasznosak a MedTech, építészet, turizmus, játékfejlesztés, marketing és még sok más területen. Ebben a leckében a mai **GPT Image** modellekkel dolgozunk a Microsoft Foundry platformon.

## Tanulási célok

A lecke végére képes leszel:

- Megmagyarázni, mi az a képgenerálás és hol hasznos.
- Megérteni a `gpt-image` modellcsaládot és hogy miben különbözik a régebbi DALL·E modellektől.
- Képgeneráló alkalmazást építeni és képeket szerkeszteni.

## Mi a képgenerálás?

A képgeneráló modellek szöveges utasítás alapján hoznak létre képeket. A modern modellek, mint például a `gpt-image`, a tanulás során megtanulják a szöveg és a képek közti kapcsolatot, majd iteratívan alakítanak véletlenszerű zajból olyan képet, ami megfelel az utasításnak.

Két jól ismert képgeneráló modellcsalád:

- **`gpt-image` (OpenAI)** — a jelenlegi generáció, amit ennél a leckénél használunk. Támogatja a szövegből kép generálást és a képszerkesztést (maszkolt inpainting).
- **Midjourney** — egy népszerű harmadik féltől származó modell, saját szolgáltatással és Discord alapú munkafolyamattal.

> A régebbi OpenAI képgeneráló modellek — **DALL·E 2** és **DALL·E 3** — elavultak. A DALL·E 3 már nem elérhető új telepítésekhez, és olyan funkciók, mint a `create_variation` csak a DALL·E 2-ben voltak. Új alkalmazásokhoz a `gpt-image` modelleket használd.

A Microsoft Foundry-n a **`gpt-image-2`** a legújabb és legerősebb képgeneráló modell, ez ajánlott alapértelmezettként. A `gpt-image-1.5` és a `gpt-image-1-mini` is általánosan elérhető.

> **Fontos:** A `gpt-image` modellek a generált képet **base64** formátumban adják vissza (`b64_json`), nem URL-ként. A kódod dekódolja a base64 karakterláncot bájtokra és elmenti — nincs letölthető kép URL.


## Az első képgeneráló alkalmazásod elkészítése

Szóval mit igényel egy képgeneráló alkalmazás elkészítése? A következő könyvtárakra van szükséged:

- **python-dotenv**, nagyon ajánlott ezt a könyvtárat használni, hogy a titkos adatokat egy *.env* fájlban tárold, távol a kódtól.
- **openai**, ezt a könyvtárat fogod használni az OpenAI API-val való kommunikációhoz.
- **pillow**, a képek kezeléséhez Pythonban.

Ha még nem tetted meg, kövesd a [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst) oldalon található utasításokat egy Microsoft Foundry erőforrás és modell létrehozásához. Válaszd ki a **gpt-image-2** modellt (a legújabb Azure OpenAI képgeneráló modell; a DALL·E már elavult).

1. Hozz létre egy *.env* fájlt a következő tartalommal:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    Ezt az információt megtalálhatod a Microsoft Foundry portálon az erőforrásod "Deployments" szekciójában.



1. Gyűjtsd össze a fenti könyvtárakat egy *requirements.txt* nevű fájlba így:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Ezután hozz létre egy virtuális környezetet, és telepítsd a könyvtárakat:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Windows esetén az alábbi parancsokat használja a virtuális környezet létrehozásához és aktiválásához:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Adja hozzá a következő kódot egy *app.py* nevű fájlba:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # importálja a dotenv-et
    dotenv.load_dotenv()

    # Konfigurálja az Azure OpenAI (Microsoft Foundry) klienst.
    # A képi modellekhez friss API verzió szükséges – ellenőrizze a Foundry dokumentációját a modellje által megkövetelt verzióért.
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # Készítsen képet a képalkotó API használatával
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Adja meg a kérés szövegét itt
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # pl. "gpt-image-2"
        )
        # Állítsa be a tárolt kép könyvtárát
        image_dir = os.path.join(os.curdir, 'images')

        # Ha a könyvtár nem létezik, hozza létre
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Inicializálja a képfájlt (figyeljen rá, hogy a fájltípus png legyen)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # A gpt-image modellek a képet base64 (b64_json) formátumban adják vissza, nem URL-ként
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Jelenítse meg a képet az alapértelmezett képnézegetőben
        image = Image.open(image_path)
        image.show()

    # Fogja el a kivételeket
    except BadRequestError as err:
        print(err)

    ```

Magyarázzuk meg ezt a kódot:

- Először importáljuk a szükséges könyvtárakat, beleértve az OpenAI könyvtárat, a dotenv könyvtárat, a base64 modult és a Pillow könyvtárat.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- Ezután betöltjük a környezeti változókat a *.env* fájlból.

    ```python
    # dotenv importálása
    dotenv.load_dotenv()
    ```

- Ezután konfiguráljuk az Azure OpenAI (Microsoft Foundry) klienst.

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- Ezután létrehozzuk a képet:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Írja be ide a kért szöveget
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    A `gpt-image` modellek a képet **base64** -es karakterláncként adják vissza a `data[0].b64_json` mezőben. Ezt bájtokká dekódoljuk, és fájlba írjuk — nincs URL a letöltéshez.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Végül megnyitjuk a képet, és a szabványos képnézegetővel jelenítjük meg:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### További részletek a kép generálásáról

Nézzük meg az `images.generate` paramétereit:

- **prompt**, az a szöveges utasítás, amely alapján a képet generáljuk. Itt a "Nyuszi lovon, nyalókát tart, egy ködös mezőn, ahol nárciszok nőnek".
- **size**, a generált kép mérete (például `1024x1024`, `1536x1024`, `1024x1536` vagy `"auto"`).
- **n**, a generált képek száma. Itt egyet generálunk.
- **model**, az Ön képgeneráló modelljének telepítési neve (például `gpt-image-2`).

> A képgeneráló modellek nem fogadnak el `temperature` paramétert — ez szöveggenerálási vezérlő. A változatosság növeléséhez hívja meg újra az API-t; a változatosság csökkentéséhez tegye az utasítást specifikusabbá.

## A képgenerálás további képességei

Már látta, hogyan lehet néhány sor Python-kóddal képet generálni. A `gpt-image` modellek képesek meglévő képet is **szerkeszteni**. Ha megad egy képet, egy opcionális **maszkot** (amely megjelöli a módosítandó területet) és egy utasítást, a kép egy részét megváltoztathatja — például kalapot tehet a nyuszira.

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# a szerkesztések szintén base64-ként kerülnek visszaadásra
edited_image = base64.b64decode(response.data[0].b64_json)
```

Az alap kép csak a nyuszit tartalmazza; a végső képen a nyuszira kalap került.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordítási szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár az pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Fontos információk esetén professzionális emberi fordítást javasolunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely ebből a fordításból ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
